In [4]:
"""
Cardinality-constrained portfolio optimizer via PySCIPOpt (direct SCIP bindings).

Solver choice: PySCIPOpt over pyomo/pulp — no adapter layer, full SCIP parameter
access, and a single extra dependency (pyscipopt) instead of two.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from pyscipopt import Model, quicksum


def optimize_portfolio(
    scenario_matrix: pd.DataFrame,   # shape (N, m), index=assets, columns=scenarios
    cost_vector: pd.Series,          # length N — transaction cost per unit |weight|
    return_vector: pd.Series,        # length N — expected return per unit weight
    const_vector: pd.Series,         # length m — lower bound on each scenario exposure
    n: int,                          # max number of assets to select (≥ 1)
    ratio: float,                    # max(|w_i|) / min(|w_i|) ≤ ratio, selected assets (≥ 1)
    bound: pd.DataFrame,             # columns ["lb","ub"], index=assets; lb may be negative
    verbose: bool = False,
) -> dict:
    """
    Maximize net portfolio return subject to cardinality, ratio, and scenario constraints.

    Objective (maximise):
        Σ_i ( return_i · w_i  −  cost_i · |w_i| )

    Constraints:
        1. Σ_i z_i ≤ n                            (cardinality)
        2. lb_i · z_i ≤ w_i ≤ ub_i · z_i         (big-M selection link; w_i=0 if z_i=0)
        3. Σ_i S[i,k] · w_i ≥ const_k  ∀k        (scenario exposure floors)
        4. max(|w_i| : z_i=1) ≤ ratio · min(|w_i| : z_i=1)  (concentration ratio)

    Returns
    -------
    dict with keys:
        status      – SCIP status string (e.g. "optimal", "infeasible")
        objective   – optimal objective value (np.nan if unsolved)
        weights     – pd.Series of w_i (0.0 for unselected assets)
        selected    – list of asset names where z_i = 1
        exposures   – scenario_matrix.T @ weights  (pd.Series indexed by scenario)
        n_selected  – int
    """
    # ── Input validation ────────────────────────────────────────────────────
    assets    = scenario_matrix.index
    scenarios = scenario_matrix.columns

    for vec, name in [(cost_vector, "cost_vector"), (return_vector, "return_vector")]:
        if not assets.equals(vec.index):
            raise ValueError(f"{name}.index must match scenario_matrix.index")
    if not assets.equals(bound.index):
        raise ValueError("bound.index must match scenario_matrix.index")
    if not scenarios.equals(const_vector.index):
        raise ValueError("const_vector.index must match scenario_matrix.columns")
    if set(bound.columns.tolist()) != {"lb", "ub"}:
        raise ValueError('bound must have exactly columns ["lb", "ub"]')
    if not (bound["lb"] < bound["ub"]).all():
        raise ValueError("bound requires lb < ub for every asset")
    if n < 1:
        raise ValueError("n must be ≥ 1")
    if ratio < 1.0:
        raise ValueError("ratio must be ≥ 1")

    # ── Model setup ─────────────────────────────────────────────────────────
    model = Model("portfolio")
    if not verbose:
        model.hideOutput()

    asset_list    = assets.tolist()
    scenario_list = scenarios.tolist()

    # Per-asset big-M: tightest bound on |w_i|, derived from input bounds
    M: dict[str, float] = {
        i: max(abs(float(bound.loc[i, "lb"])), abs(float(bound.loc[i, "ub"])))
        for i in asset_list
    }
    M_global = max(M.values())

    # ── Decision variables ───────────────────────────────────────────────────
    w: dict = {}   # continuous weight (negative allowed when lb < 0 — short)
    z: dict = {}   # binary selection indicator
    a: dict = {}   # |w_i| auxiliary (linearised absolute value)

    for i in asset_list:
        lb_i = float(bound.loc[i, "lb"])
        ub_i = float(bound.loc[i, "ub"])
        Mi   = M[i]
        w[i] = model.addVar(lb=lb_i, ub=ub_i, vtype="C", name=f"w_{i}")
        z[i] = model.addVar(lb=0,    ub=1,     vtype="B", name=f"z_{i}")
        # ub=Mi on a[i] is a valid global bound; the abs_ub constraint below
        # tightens it to 0 when z_i=0, which also strengthens the LP relaxation.
        a[i] = model.addVar(lb=0,    ub=Mi,    vtype="C", name=f"a_{i}")

    # Ratio auxiliaries:
    #   w_max tracks max(|w_i| : z_i=1) via  a_i ≤ w_max  (when z_i=1)
    #   w_min tracks min(|w_i| : z_i=1) via  a_i ≥ w_min  (when z_i=1)
    w_max = model.addVar(lb=0, ub=M_global, vtype="C", name="w_max")
    w_min = model.addVar(lb=0, ub=M_global, vtype="C", name="w_min")

    # ── Objective ────────────────────────────────────────────────────────────
    model.setObjective(
        quicksum(
            float(return_vector[i]) * w[i] - float(cost_vector[i]) * a[i]
            for i in asset_list
        ),
        "maximize",
    )

    # ── Constraints ──────────────────────────────────────────────────────────

    # 1. Cardinality: at most n assets
    model.addCons(quicksum(z[i] for i in asset_list) <= n, name="cardinality")

    for i in asset_list:
        lb_i = float(bound.loc[i, "lb"])
        ub_i = float(bound.loc[i, "ub"])
        Mi   = float(M[i])

        # 2. Big-M selection link (handles both long-only and long-short bounds)
        #    When z_i=0: lb_i·0=0 ≤ w_i and w_i ≤ ub_i·0=0  →  w_i = 0
        #    When z_i=1: lb_i ≤ w_i ≤ ub_i  (asset's own bounds are active)
        model.addCons(w[i] >= lb_i * z[i], name=f"sel_lb_{i}")
        model.addCons(w[i] <= ub_i * z[i], name=f"sel_ub_{i}")

        # 3. Linearise |w_i| via envelope: a_i ≥ w_i and a_i ≥ -w_i
        model.addCons(a[i] >= w[i],      name=f"abs_pos_{i}")
        model.addCons(a[i] >= -w[i],     name=f"abs_neg_{i}")
        # Enforce a_i = 0 when z_i = 0 (implied by sel_lb/ub, but stated
        # explicitly to tighten the LP relaxation and ratio big-M activations)
        model.addCons(a[i] <= Mi * z[i], name=f"abs_ub_{i}")

        # 4. Ratio auxiliary constraints (big-M deactivated when z_i = 0):
        #    When z_i=1: a_i ≤ w_max  →  w_max is an upper bound on selected |w|
        model.addCons(a[i] <= w_max + Mi * (1 - z[i]), name=f"rmax_{i}")
        #    When z_i=1: a_i ≥ w_min  →  w_min is a lower bound on selected |w|
        #    When z_i=0 and no asset selected: w_min stays 0 (lb enforced by var)
        model.addCons(a[i] >= w_min - Mi * (1 - z[i]), name=f"rmin_{i}")

    # 4b. Ratio cap: max|w| ≤ ratio · min|w| (vacuous when all z_i=0 since both = 0)
    model.addCons(w_max <= ratio * w_min, name="ratio_cap")

    # 5. Scenario exposure floors: Σ_i S[i,k]·w_i ≥ const_k
    for k in scenario_list:
        model.addCons(
            quicksum(float(scenario_matrix.loc[i, k]) * w[i] for i in asset_list)
            >= float(const_vector[k]),
            name=f"exposure_{k}",
        )

    # ── Solve ─────────────────────────────────────────────────────────────────
    model.optimize()
    status = model.getStatus()

    _infeasible_result = dict(
        status=status,
        objective=np.nan,
        weights=pd.Series(0.0, index=assets),
        selected=[],
        exposures=pd.Series(0.0, index=scenarios),
        n_selected=0,
    )

    # "bestsol" = a feasible incumbent exists but optimality not proven (e.g. time limit)
    if status not in ("optimal", "bestsol"):
        return _infeasible_result

    weights = pd.Series(
        {i: model.getVal(w[i]) for i in asset_list}, dtype=float
    )
    weights[weights.abs() < 1e-10] = 0.0
    selected = [i for i in asset_list if model.getVal(z[i]) > 0.5]

    return {
        "status":     status,
        "objective":  float(model.getObjVal()),
        "weights":    weights,
        "selected":   selected,
        "exposures":  scenario_matrix.T @ weights,
        "n_selected": len(selected),
    }


# ── Synthetic example ─────────────────────────────────────────────────────────
if __name__ == "__main__":
    rng = np.random.default_rng(42)
    N, m = 10, 3
    assets    = [f"A{i:02d}" for i in range(N)]
    scenarios = ["S0", "S1", "S2"]

    scenario_matrix = pd.DataFrame(
        rng.uniform(-1.0, 1.0, (N, m)), index=assets, columns=scenarios
    )
    return_vector = pd.Series(rng.uniform(-0.02, 0.12, N), index=assets)
    cost_vector   = pd.Series(rng.uniform(0.001, 0.005, N), index=assets)
    # Loose negative floors — easy to satisfy so the example is reliably feasible
    const_vector  = pd.Series([-2.0, -2.0, -2.0], index=scenarios)
    bound = pd.DataFrame(
        {"lb": [-0.15] * N, "ub": [0.25] * N}, index=assets
    )

    result = optimize_portfolio(
        scenario_matrix=scenario_matrix,
        cost_vector=cost_vector,
        return_vector=return_vector,
        const_vector=const_vector,
        n=2,
        ratio=3.0,
        bound=bound,
        verbose=False,
    )

    print(f"\nStatus     : {result['status']}")
    print(f"Objective  : {result['objective']:.6f}")
    print(f"n_selected : {result['n_selected']}")
    print(f"Selected   : {result['selected']}")
    print(f"\nWeights:\n{result['weights'][result['weights'] != 0].round(6)}")
    print(f"\nExposures:\n{result['exposures'].round(6)}")

    # Verify ratio constraint
    sel_w = result['weights'][result['selected']].abs()
    if len(sel_w) > 0 and sel_w.min() > 1e-10:
        actual_ratio = sel_w.max() / sel_w.min()
        print(f"\nActual max/min |w| ratio: {actual_ratio:.4f}  (limit = 3.0)")



Status     : optimal
Objective  : 0.048160
n_selected : 2
Selected   : ['A00', 'A01']

Weights:
A00    0.25
A01    0.25
dtype: float64

Exposures:
S0    0.235662
S1   -0.233472
S2    0.417110
dtype: float64

Actual max/min |w| ratio: 1.0000  (limit = 3.0)


In [5]:
scenario_matrix

,S0,S1,S2
A00,0.547912,-0.122243,0.717196
A01,0.394736,-0.811645,0.951245
A02,0.522279,0.572129,-0.743773
A03,-0.099228,-0.258404,0.853530
A04,0.287730,0.645523,-0.113172
A05,-0.545523,0.109170,-0.872365
A06,0.655262,0.263329,0.516175
A07,-0.290948,0.941396,0.786242
A08,0.556767,-0.610723,-0.066558
A09,-0.912392,-0.691421,0.366098


In [6]:
cost_vector

A00    0.002749
A01    0.004331
A02    0.003801
A03    0.002249
A04    0.004329
A05    0.004219
A06    0.002550
A07    0.002153
A08    0.003730
A09    0.001559
dtype: float64

In [7]:
return_vector

A00    0.084267
A01    0.115451
A02    0.025616
A03    0.031864
A04    0.045738
A05    0.006526
A06   -0.001811
A07    0.046599
A08    0.011767
A09    0.073774
dtype: float64